# Практическая работа №4. Сравнение наборов признаков

Целевая переменная — `pulled_count` (общее число полученных копий персонажа). В модель включаются только числовые признаки: столбец `Unnamed: 0` исключается как технический индекс, а сама целевая переменная — чтобы не допустить утечку.

Модели сравниваются на одном и том же отложенном тестовом наборе. Мультиколлинеарность оценивается по корреляциям, вычисленным только на обучающей части данных.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

In [ ]:
data_path = Path('../../data/genshin_impact.csv')
if not data_path.exists():
    data_path = Path('data/genshin_impact.csv')

df = pd.read_csv(data_path)
print(f'Размер датасета: {df.shape[0]} строк, {df.shape[1]} столбцов')
print(f'Пропусков: {int(df.isna().sum().sum())}')
df.head()

## 1. Базовая модель на всех доступных числовых признаках

In [ ]:
target = 'pulled_count'
technical_columns = ['Unnamed: 0']
numeric_features = [
    column
    for column in df.select_dtypes(include='number').columns
    if column not in [target, *technical_columns]
]

X = df[numeric_features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Числовых признаков в базовой модели: {len(numeric_features)}')
print(numeric_features)
print(f'Обучающая выборка: {X_train.shape[0]} наблюдений; тестовая: {X_test.shape[0]}')

In [ ]:
def make_linear_model():
    return Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', LinearRegression()),
    ])

def regression_metrics(model_name, y_true, y_pred):
    return {
        'Модель': model_name,
        'R²': r2_score(y_true, y_pred),
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
    }

baseline_model = make_linear_model()
baseline_model.fit(X_train, y_train)
baseline_predictions = baseline_model.predict(X_test)

baseline_metrics = regression_metrics(
    'Базовая модель: все признаки', y_test, baseline_predictions
)
pd.DataFrame([baseline_metrics]).set_index('Модель').round(4)

## 2. Проверка мультиколлинеарности и сокращение набора признаков

Порог сильной корреляции принимается равным `|r| ≥ 0.85`. При выборе представителя кластера оставляется признак с наибольшей по модулю корреляцией с целью на обучающей выборке. Такой выбор не использует тестовые ответы.

In [ ]:
correlation_threshold = 0.85
train_correlation = X_train[numeric_features].corr()
target_correlation = X_train.assign(**{target: y_train})[numeric_features + [target]].corr()[target].drop(target)

fig, ax = plt.subplots(figsize=(15, 12))
image = ax.imshow(train_correlation, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_features)), numeric_features, rotation=90)
ax.set_yticks(range(len(numeric_features)), numeric_features)
for row in range(len(numeric_features)):
    for column in range(len(numeric_features)):
        ax.text(column, row, f'{train_correlation.iloc[row, column]:.2f}',
                ha='center', va='center', fontsize=7)
fig.colorbar(image, ax=ax, label='Коэффициент корреляции Пирсона')
ax.set_title('Корреляционная матрица признаков (обучающая выборка)')
plt.tight_layout()
plt.show()

In [ ]:
upper_triangle = train_correlation.where(
    np.triu(np.ones(train_correlation.shape), k=1).astype(bool)
)
high_correlation_pairs = (
    upper_triangle.stack()
    .rename('r')
    .reset_index()
    .rename(columns={'level_0': 'Признак 1', 'level_1': 'Признак 2'})
)
high_correlation_pairs['|r|'] = high_correlation_pairs['r'].abs()
high_correlation_pairs = high_correlation_pairs[high_correlation_pairs['|r|'] >= correlation_threshold]
high_correlation_pairs = high_correlation_pairs.sort_values('|r|', ascending=False)

print('Сильные корреляции:')
high_correlation_pairs.round(4)

In [ ]:
print('Корреляции признаков с целевой переменной на обучающей выборке:')
target_correlation.sort_values(key=lambda values: values.abs(), ascending=False).to_frame('r с pulled_count').round(4)

In [ ]:
# Группы избыточных признаков, найденные по корреляционной матрице.
redundant_groups = {
    'Время с момента выпуска': ['days_since_release', 'months_since_release'],
    'Повторные показы и constellations': [
        'constellation_1_pull', 'constellation_2_pull', 'constellation_3_pull',
        'constellation_4_pull', 'constellation_5_pull', 'constellation_6_pull',
        'constellation_7_pull', 'num_banners'
    ],
    'Редкость и дубликаты': [
        'rarity', 'duplicate_rate', 'avg_copies_per_player', 'is_standard_banner'
    ],
}

representatives = {
    group_name: max(group, key=lambda feature: abs(target_correlation[feature]))
    for group_name, group in redundant_groups.items()
}
# Для полностью дублирующих признаков выбираем исходную величину в днях.
representatives['Время с момента выпуска'] = 'days_since_release'

# constellation_0_pull не входит в сильный корреляционный кластер при |r| >= 0.85,
# поэтому сохраняется как отдельный информативный признак.
reduced_features = [
    representatives['Редкость и дубликаты'],
    representatives['Время с момента выпуска'],
    'constellation_0_pull',
    representatives['Повторные показы и constellations'],
    'lvl_90_HP', 'lvl_90_ATK', 'lvl_90_DEF',
    'c6_rate', 'is_archon',
]

removed_features = [feature for feature in numeric_features if feature not in reduced_features]
print('Представители корреляционных кластеров:')
print(representatives)
print(f'\nСокращённый набор ({len(reduced_features)} признаков):')
print(reduced_features)
print(f'\nУдалены как избыточные ({len(removed_features)} признаков):')
print(removed_features)

In [ ]:
reduced_correlation = X_train[reduced_features].corr().abs()
reduced_upper = reduced_correlation.where(
    np.triu(np.ones(reduced_correlation.shape), k=1).astype(bool)
)
remaining_max_correlation = reduced_upper.stack().sort_values(ascending=False).head(1)
print('Максимальная парная корреляция в сокращённом наборе:')
remaining_max_correlation.round(4)

## 3. Модель на сокращённом наборе и сравнение метрик

In [ ]:
reduced_model = make_linear_model()
reduced_model.fit(X_train[reduced_features], y_train)
reduced_predictions = reduced_model.predict(X_test[reduced_features])

reduced_metrics = regression_metrics(
    'Сокращённая модель', y_test, reduced_predictions
)
comparison = pd.DataFrame([baseline_metrics, reduced_metrics]).set_index('Модель')
comparison.round({'R²': 4, 'MAE': 2, 'RMSE': 2})

Базовая модель показывает почти идеальный результат, но это объясняется структурой данных: `pulled_count` практически равен сумме столбцов `constellation_*_pull`. Поэтому такой результат нельзя трактовать как доказательство самостоятельной полезности всех 20 признаков. Сокращённая модель сохраняет высокую точность при меньшем числе признаков и меньшей избыточности.

## 4. График остатков и проверка систематического паттерна

In [ ]:
residuals = y_test.to_numpy() - reduced_predictions
residual_correlation = np.corrcoef(reduced_predictions, residuals)[0, 1]
absolute_residual_correlation = np.corrcoef(reduced_predictions, np.abs(residuals))[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(reduced_predictions, residuals, color='tab:blue', edgecolor='white', s=70)
axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0].set_title('Остатки сокращённой модели')
axes[0].set_xlabel('Предсказанное значение pulled_count')
axes[0].set_ylabel('Остаток: y - ŷ')
axes[0].ticklabel_format(style='sci', axis='both', scilimits=(0, 0))
axes[0].grid(alpha=0.3)

relative_residuals = residuals / y_test.to_numpy()
axes[1].scatter(reduced_predictions, relative_residuals, color='tab:orange', edgecolor='white', s=70)
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Относительные остатки')
axes[1].set_xlabel('Предсказанное значение pulled_count')
axes[1].set_ylabel('(y - ŷ) / y')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Корреляция предсказаний с остатками: {residual_correlation:.3f}')
print(f'Корреляция предсказаний с модулем остатков: {absolute_residual_correlation:.3f}')
if abs(residual_correlation) < 0.2:
    print('Вывод: выраженного линейного тренда в остатках не наблюдается.')
else:
    print('Вывод: возможен систематический тренд в остатках; линейная спецификация требует осторожной интерпретации.')

## 5. Итоговый вывод

Финальной выбрана линейная модель на сокращённом наборе из 9 признаков: `rarity`, `days_since_release`, `constellation_0_pull`, `constellation_1_pull`, `lvl_90_HP`, `lvl_90_ATK`, `lvl_90_DEF`, `c6_rate`, `is_archon`.

Из модели удалены точный дубликат времени (`months_since_release`), остальные сильно коррелированные уровни `constellation`, а также признаки, дублирующие информацию о редкости и количестве повторных показов. При этом `constellation_0_pull` оставлен: его корреляция с `constellation_1_pull` ниже порога 0.85, и он заметно улучшает качество по сравнению с моделью, где оставлен только один признак из этого блока.

Сокращённая модель немного уступает базовой по метрикам, но сохраняет высокий `R²` и существенно проще для интерпретации. На графике остатков выраженного линейного паттерна нет; однако из-за небольшого размера выборки и разного масштаба `pulled_count` вывод о гомоскедастичности следует считать предварительным.